In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
pip install nltk

In [ ]:
pip install transformers torch --quiet

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag
from nltk.util import ngrams
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from transformers import BertTokenizer, BertModel
import torch
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'colab'

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('stopwords')

In [ ]:
full_data=pd.read_csv('drive/MyDrive/ML_course/fake_news_full_data.csv', index_col=0)

In [ ]:
data=full_data.sample(frac=0.2, random_state=43)

#EDA

In [ ]:
data

In [ ]:
data.info()

In [ ]:
full_data.info()

In [ ]:
data.drop(15914, axis=0, inplace=True)

In [ ]:
data['date']=pd.to_datetime(data['date'], format='mixed', dayfirst=True)

In [ ]:
data.is_fake.value_counts(normalize=True)

In [ ]:
data['date'].min(), data['date'].max()

In [ ]:
data[data['is_fake']==1]['date'].min(), data[data['is_fake']==1]['date'].max()

In [ ]:
data[data['is_fake']==0]['date'].min(), data[data['is_fake']==0]['date'].max()

In [ ]:
data['len_title']=data['title'].str.len()

In [ ]:
data['len_text']=data['text'].str.len()

In [ ]:
data.groupby('is_fake')[['len_title','len_text']].describe().T.stack().reset_index()

In [ ]:
a=data.groupby('is_fake')[['len_title','len_text']].describe().T.stack().reset_index()

In [ ]:
px.histogram(data_frame=a[a['level_0']=='len_title'], x='level_1', y=0, color='is_fake', barmode='group', barnorm='percent')

In [ ]:
px.histogram(data_frame=a[a['level_0']=='len_text'], x='level_1', y=0, color='is_fake', barmode='group', barnorm='percent')

In [ ]:
data[(data['len_text']==1)]['is_fake'].value_counts()

In [ ]:
lens_text=data[['len_text','is_fake']].value_counts().reset_index().sort_values(by=['len_text','is_fake'])

In [ ]:
px.line(lens_text[(lens_text['len_text']>43)&(lens_text['len_text']<10000)], x='len_text', y='count', facet_row='is_fake')

In [ ]:
lens_title=data[['len_title','is_fake']].value_counts().reset_index().sort_values(by=['len_title','is_fake'])

In [ ]:
px.line(lens_title, x='len_title', y='count', facet_row='is_fake')

#Text Preprocessing

In [ ]:
english_stopwords=stopwords.words('english')
extra_tokens=["'d", "'ll", "'m", "'re", "'s", "'ve", 'could', 'might', 'must', "n't", 'need', 'sha', 'wo', 'would']
english_stopwords.extend(extra_tokens)

In [ ]:
# english_stopwords=[i for i in english_stopwords if i not in ["aren't",'couldn',
#  "couldn't",'didn',
#  "didn't", 'doesn',
#  "doesn't",'don',
#  "don't",'hadn',
#  "hadn't",'hasn',
#  "hasn't",'haven',
#  "haven't", 'isn',
#  "isn't",'mightn',
#  "mightn't",'needn',
#  "needn't",'no',
#  'not','shan',
#  "shan't",'shouldn',
#  "shouldn't",'wasn',
#  "wasn't",'weren',
#  "weren't","won't",
#  'wouldn',
#  "wouldn't",]]

In [ ]:
def tokenize(text):
  return [word for word in word_tokenize(text.lower()) if word not in english_stopwords]

In [ ]:
def get_tag(text):
  tokens=tokenize(text)
  tags=pos_tag(tokens)
  return tags

In [ ]:
def get_wordnet_pos(treebank_tag):

    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

In [ ]:
lem=WordNetLemmatizer()
def lemmatization(text):
  tags=get_tag(text)
  lemmatized_words=[]
  for word, tag in tags:
    wordnet_tag=get_wordnet_pos(tag)
    if wordnet_tag:
      lemma=lem.lemmatize(word, wordnet_tag)
    else:
      lemma=word
    lemmatized_words.append(lemma)
  return lemmatized_words


In [ ]:
train_idx, test_idx=train_test_split(list(data.index), test_size=0.2, shuffle=True)

In [ ]:
data['day']=data['date'].dt.day
data['month']=data['date'].dt.month
data['year']=data['date'].dt.year
data['dayofweek']=data['date'].dt.dayofweek

In [ ]:
data.drop('date', axis=1, inplace=True)

In [ ]:
train_inputs=data.loc[train_idx].drop('is_fake', axis=1)
train_target=data.loc[train_idx]['is_fake']

In [ ]:
test_inputs=data.loc[test_idx].drop('is_fake', axis=1)
test_target=data.loc[test_idx]['is_fake']

In [ ]:
tfidf_text=TfidfVectorizer(lowercase=True,
                           tokenizer=lemmatization,
                           stop_words=english_stopwords,
                           max_features=1000)

In [ ]:
tfidf_title=TfidfVectorizer(lowercase=True,
                           tokenizer=lemmatization,
                           stop_words=english_stopwords,
                           max_features=1000)

In [ ]:
text_train=pd.DataFrame(tfidf_text.fit_transform(train_inputs['text']).toarray(), columns=tfidf_text.get_feature_names_out(), index=train_inputs.index)
text_test=pd.DataFrame(tfidf_text.transform(test_inputs['text']).toarray(), columns=tfidf_text.get_feature_names_out(), index=test_inputs.index)

In [ ]:
title_train=pd.DataFrame(tfidf_title.fit_transform(train_inputs['title']).toarray(), columns=tfidf_title.get_feature_names_out(), index=train_inputs.index)
title_test=pd.DataFrame(tfidf_title.transform(test_inputs['title']).toarray(), columns=tfidf_title.get_feature_names_out(), index=test_inputs.index)

In [ ]:
title_train

In [ ]:
# train_set=pd.concat([train_inputs, text_train, title_train], axis=1).drop(['title', 'text'], axis=1)
# test_set=pd.concat([test_inputs, text_test, title_test], axis=1).drop(['title', 'text'], axis=1)

In [ ]:
train_set=pd.concat([ text_train, title_train], axis=1)
test_set=pd.concat([text_test, title_test], axis=1)

# Logistic Regression

In [ ]:
logreg=LogisticRegression(solver='saga', penalty='l1', max_iter=1000)

In [ ]:
def predictions(model, train_input, test_input, train_target, test_target):
  model.fit(train_input, train_target)
  pred=model.predict(test_input)
  accuracy=accuracy_score(test_target, pred)
  f1=f1_score(test_target, pred, average='weighted')
  confusion_m=confusion_matrix(test_target, pred)
  print(f'accurracy score for test dataset : {accuracy}\nf1_score for test dataset : {f1}\nconfusion matrix\n{confusion_m}' )

In [ ]:
predictions(logreg, train_set, test_set, train_target, test_target)

In [ ]:
pd.DataFrame(logreg.coef_[0], index=train_set.columns, columns=['importance']).sort_values(by='importance', ascending=False)[:30]

#Random Forest

In [ ]:
forest=RandomForestClassifier(n_estimators=100, max_depth=300, random_state=45)

In [ ]:
predictions(forest, train_set, test_set, train_target, test_target)

In [ ]:
data_sample=full_data.loc[[i for i in full_data.index if i not in train_idx and i not in test_idx]][:2000]

In [ ]:
data_sample

In [ ]:
text=pd.DataFrame(tfidf_text.transform(data_sample['text']).toarray(), columns=tfidf_text.get_feature_names_out(), index=data_sample.index)
title=pd.DataFrame(tfidf_title.transform(data_sample['title']).toarray(), columns=tfidf_title.get_feature_names_out(), index=data_sample.index)

In [ ]:
sample_test=pd.concat([text, title], axis=1)

In [ ]:
sample_target=data_sample['is_fake']

In [ ]:
predictions(forest, train_set, sample_test, train_target, sample_target)

#BERT

In [ ]:
bert_tokenizer=BertTokenizer.from_pretrained('bert-base-uncased')
bert_model=BertModel.from_pretrained('bert-base-uncased')

In [ ]:
input_bert=bert_tokenizer(test_inputs['text'].tolist(), return_tensors='pt', truncation=True, padding=True)

In [ ]:
with torch.no_grad():
  outputs=bert_model(**input_bert)

In [ ]:
input_bert